### Spider dataset analysis

In [ ]:
from collections import Counter
from itertools import combinations
from pathlib import Path
from statistics import median
from textwrap import indent
from typing import get_args

from text2sql.data import build_samples
from text2sql.prompt import build_completion, build_prompt
from text2sql.spider import SPIDER_DIR, Split, load_examples, load_schemas, query_error

Questions and databases of every split, and the schemas of all databases.

In [ ]:
spider_dir = Path("..") / SPIDER_DIR  # this notebook runs from notebooks/
splits = get_args(Split)

examples = {split: load_examples(spider_dir, split) for split in splits}
dbs = {split: {ex["db_id"] for ex in examples[split]} for split in splits}
schemas = load_schemas(spider_dir, test=True)  # test_tables.json covers all 206 databases

Prompt and completion messages the model gets for the first dev question.

In [ ]:
sample = build_samples(spider_dir, "dev")[0]

for message in build_prompt(sample) + build_completion(sample):
    print(f"{message['role']}:")
    print(indent(message["content"], "    "), end="\n\n")

Number of questions and distinct databases per split.

In [ ]:
print(f"{'split':13s} {'questions':>9s} {'databases':>9s}")
for split in splits:
    print(f"{split:13s} {len(examples[split]):9d} {len(dbs[split]):9d}")

Number of databases that appear in both splits of each pair.

In [ ]:
print(f"{'split':13s} {'split':13s} {'shared databases':>16s}")
for first, second in combinations(splits, 2):
    print(f"{first:13s} {second:13s} {len(dbs[first] & dbs[second]):16d}")

Median and max number of tables and columns per database, over the databases each split uses.

In [ ]:
print(f"{'split':13s} {'tables':>15s} {'columns':>15s}")
print(f"{'':13s} {'median':>7s} {'max':>7s} {'median':>7s} {'max':>7s}")
for split in splits:
    tables = [len(schemas[db_id]["table_names_original"]) for db_id in dbs[split]]
    columns = [
        len(schemas[db_id]["column_names_original"]) - 1 for db_id in dbs[split]
    ]  # without "*"
    print(
        f"{split:13s} {median(tables):7.1f} {max(tables):7d} {median(columns):7.1f} {max(columns):7d}"
    )

Share of gold queries that contain each SQL keyword, per split.

In [ ]:
keywords = [
    "JOIN",
    "WHERE",
    "GROUP BY",
    "HAVING",
    "ORDER BY",
    "LIMIT",
    "DISTINCT",
    "INTERSECT",
    "UNION",
    "EXCEPT",
]
queries = {
    split: [" ".join(ex["query"].upper().split()) for ex in examples[split]] for split in splits
}

print(" " * 10 + "".join(f"{split:>14s}" for split in splits))
for keyword in keywords:
    print(
        f"{keyword:10s}"
        + "".join(
            f"{sum(keyword in q for q in queries[split]) / len(queries[split]):14.0%}"
            for split in splits
        )
    )

Number of repeated questions on the same database, per split.

In [ ]:
print(f"{'split':13s} {'duplicates':>10s}")
for split in splits:
    counts = Counter((ex["db_id"], ex["question"]) for ex in examples[split])
    print(f"{split:13s} {sum(count - 1 for count in counts.values()):10d}")

Gold queries that raise an error when run on their database.

In [ ]:
print(f"{'split':13s} {'index':>5s}  {'database':24s} error")
for split in splits:
    for i, ex in enumerate(examples[split]):
        error = query_error(spider_dir, ex["db_id"], ex["query"], test=split == "test")
        if error:
            print(f"{split:13s} {i:5d}  {ex['db_id']:24s} {error}")